# Dự đoán kết quả trận đấu Ngoại hạng Anh bằng XGBoost

Notebook này sẽ đọc dữ liệu Gold (đã qua làm sạch), loại bỏ các biến gây rò rỉ dữ liệu (leakage), chia tập Train/Test theo thời gian và huấn luyện mô hình phân loại đa lớp XGBoost.

In [ ]:
# Cài đặt thư viện (Nếu chạy local, trên Colab đã có sẵn)
# !pip install xgboost scikit-learn seaborn matplotlib pandas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

# Thư viện đọc MinIO của Project
from lake import minio_io as mio


In [ ]:
# 1. THIẾT LẬP KẾT NỐI MINIO VÀ TẢI DỮ LIỆU (BẢNG GOLD)
os.environ['MINIO_ENDPOINT'] = 'http://20.41.113.183:9000'
os.environ['MINIO_ACCESS_KEY'] = 'minioadmin'
os.environ['MINIO_SECRET_KEY'] = 'minioadmin123'

print("Đang tải dữ liệu từ MinIO...")
try:
    file_bytes = mio.read_bytes('gold/features/feature_match_ml.parquet')
    df = pd.read_parquet(io.BytesIO(file_bytes))
    print(f"✅ Tải thành công! Kích thước dữ liệu: {df.shape[0]} trận đấu, {df.shape[1]} thuộc tính.")
except Exception as e:
    print(f"❌ Lỗi tải dữ liệu: {e}")
    
df.head()


In [ ]:
# 2. TIỀN XỬ LÝ (PREPROCESSING) VÀ CHỐNG RÒ RỈ DỮ LIỆU (DATA LEAKAGE)

# 2.1 Loại bỏ các cột Data Leakage (Những thứ xảy ra trong hoặc sau trận đấu)
# Các cột không được đưa vào mô hình vì nó chứa kết quả!
leakage_cols = [
    'home_goals', 'away_goals', 
    'h_shots', 'a_shots', 'h_sot', 'a_sot', 
    'h_corners', 'a_corners', 'h_yellow', 'a_yellow', 'h_red', 'a_red',
    'xg_home', 'xg_away', 'npxg_home', 'npxg_away', 'ppda_home', 'ppda_away' # XG của TỪNG trận đấu là rò rỉ. XG trung bình 5 trận (home_xg_l5) thì giữ lại!
]

# Lưu lại các cột nhận dạng
id_cols = ['match_id', 'match_date', 'season', 'division', 'home_team', 'away_team']

# Cột Target (Mục tiêu dự đoán)
target_col = 'result'

# Các cột sẽ dùng làm Feature (Bỏ id, bỏ leakage, bỏ target)
features_to_drop = [c for c in df.columns if c in leakage_cols + id_cols + [target_col]]
feature_cols = [c for c in df.columns if c not in features_to_drop]

print(f"Số lượng Features sử dụng để train: {len(feature_cols)}")

# 2.2 Mã hóa biến Mục tiêu (Target) H, D, A thành 0, 1, 2
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df[target_col])
print("Mapping kết quả:", dict(zip(le.classes_, le.transform(le.classes_))))

# Đảm bảo dữ liệu được sắp xếp theo thời gian
df = df.sort_values('match_date').reset_index(drop=True)


In [ ]:
# 3. CHIA TẬP TRAIN / TEST THEO THỜI GIAN
# Mô hình dự đoán tương lai nên ta dùng mùa cũ để Train, mùa mới để Test
# Hoặc đơn giản là dùng 80% trận cũ nhất Train, 20% trận mới nhất Test.

# Chỉ chọn các dòng có đủ dữ liệu ở các feature quan trọng (Drop NaN)
df_clean = df.dropna(subset=feature_cols)
print(f"Số trận đấu sau khi bỏ Missing Values: {len(df_clean)}")

X = df_clean[feature_cols]
y = df_clean['target_encoded']

# Split theo thứ tự thời gian (không shuffle)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Kích thước Train: {X_train.shape}")
print(f"Kích thước Test: {X_test.shape}")


In [ ]:
# 4. HUẤN LUYỆN MÔ HÌNH XGBOOST

print("Bắt đầu huấn luyện XGBoost Classifier...")
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    objective='multi:softprob',
    eval_metric='mlogloss'
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=10 # In log mỗi 10 vòng
)
print("✅ Huấn luyện hoàn tất!")


In [ ]:
# 5. ĐÁNH GIÁ MÔ HÌNH (EVALUATION)

# Dự đoán trên tập Test
y_pred = xgb_model.predict(X_test)

# Đánh giá Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy trên tập Test: {acc*100:.2f}%\n")

# Báo cáo phân loại
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Vẽ Ma trận nhầm lẫn (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix")
plt.ylabel('Actual Result')
plt.xlabel('Predicted Result')
plt.show()


In [ ]:
# 6. MỨC ĐỘ QUAN TRỌNG CỦA CÁC ĐẶC TRƯNG (FEATURE IMPORTANCE)

# Trích xuất tầm quan trọng của các biến
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Vẽ Top 20 Features
plt.figure(figsize=(10, 8))
sns.barplot(data=importance.head(20), x='Importance', y='Feature', palette='viridis')
plt.title("Top 20 Tính năng quan trọng nhất (XGBoost Feature Importance)")
plt.xlabel("Mức độ quan trọng")
plt.grid(alpha=0.3)
plt.show()
